# PhoBERT Official PEFT Benchmark

Notebook này chạy benchmark Full Fine-Tuning, LoRA và DoRA official PEFT cho PhoBERT trên UIT-ViON. Kết quả được ghi vào `results/offical_benchmark_result.csv` với schema giống `results/benchmark_results_2.csv`.

In [ ]:
from pathlib import Path
import csv
import inspect
import itertools
import json
import os
import platform
import random
import sys
import time
from typing import Any

if sys.platform.startswith('win'):
    import asyncio
    asyncio.set_event_loop_policy(asyncio.WindowsSelectorEventLoopPolicy())

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

RESULTS = ROOT / 'results' / 'offical_benchmark_result.csv'
REFERENCE_RESULTS = ROOT / 'results' / 'benchmark_results_2.csv'
OUTPUT_ROOT = ROOT / 'outputs'
MODEL_NAME = 'vinai/phobert-base-v2'
DATASET = 'data/uit_vion/subset.csv'
DATASET_PATH = ROOT / DATASET
TARGET_MODULES = ['query', 'value']
RESULT_COLUMNS = [
    'run_id', 'method', 'model_name', 'dataset', 'task_type', 'rank', 'alpha', 'dropout', 'seed',
    'init_magnitude', 'use_detached_gradient', 'accuracy', 'micro_f1', 'macro_f1', 'weighted_f1',
    'samples_f1', 'hamming_loss', 'label_threshold', 'trainable_params', 'total_params',
    'trainable_percent', 'train_time_sec', 'peak_vram_mb', 'checkpoint_size_mb', 'output_dir',
]

ROOT, RESULTS

## Dependency Check

In [ ]:
import peft
from peft import LoraConfig, TaskType, get_peft_model

version_parts = tuple(int(part) for part in peft.__version__.split('.')[:2])
assert version_parts >= (0, 19), f'Need peft>=0.19 for official DoRA support, found {peft.__version__}'
assert 'use_dora' in inspect.signature(LoraConfig.__init__).parameters
print('PEFT:', peft.__version__)
print('PEFT path:', peft.__file__)
print('Official DoRA available:', 'use_dora' in inspect.signature(LoraConfig.__init__).parameters)

## GPU Diagnostics

In [ ]:
import torch

print('Python:', sys.executable)
print('Platform:', platform.platform())
print('PyTorch:', torch.__version__)
print('PyTorch CUDA build:', torch.version.cuda)
print('CUDA available:', torch.cuda.is_available())
print('CUDA device count:', torch.cuda.device_count())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    major, minor = torch.cuda.get_device_capability(0)
    print('Compute capability:', f'{major}.{minor}')
else:
    print('This environment is using CPU-only PyTorch or cannot access CUDA.')
    print('Install a CUDA-enabled PyTorch wheel in this exact environment:')
    print(f'{sys.executable} -m pip install --upgrade --index-url https://download.pytorch.org/whl/cu129 torch torchvision torchaudio')

## Download Model and Dataset

In [ ]:
DOWNLOAD_ASSETS = False
DATASET_NAME = 'uitnlp/vietnamese_students_feedback'
LOCAL_FILES_ONLY = False
FORCE_DOWNLOAD = False

if DOWNLOAD_ASSETS:
    os.environ.setdefault('HF_HUB_DISABLE_SYMLINKS_WARNING', '1')
    from datasets import load_dataset
    from huggingface_hub import snapshot_download
    from transformers import AutoConfig, AutoModelForSequenceClassification, AutoTokenizer

    model_cache_dir = snapshot_download(
        repo_id=MODEL_NAME,
        local_files_only=LOCAL_FILES_ONLY,
        force_download=FORCE_DOWNLOAD,
    )
    tokenizer = AutoTokenizer.from_pretrained(model_cache_dir, use_fast=False, local_files_only=True)
    config = AutoConfig.from_pretrained(model_cache_dir, num_labels=3, local_files_only=True)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_cache_dir,
        config=config,
        local_files_only=True,
        ignore_mismatched_sizes=True,
    )
    dataset = load_dataset(DATASET_NAME, trust_remote_code=True)
    print('PhoBERT cache:', model_cache_dir)
    print('Tokenizer vocab size:', len(tokenizer))
    print('Model loaded:', type(model).__name__)
    print(dataset)
else:
    print('Set DOWNLOAD_ASSETS = True to download PhoBERT and UIT-VSFC into the local Hugging Face cache.')

## Training Helpers

In [ ]:
import numpy as np
from transformers import AutoTokenizer, DataCollatorWithPadding, Trainer, TrainingArguments

from src.models import build_phobert_classifier
from src.peft import count_parameters
from train import (
    compute_metrics,
    directory_size_mb,
    load_named_or_local_dataset,
    make_multilabel_compute_metrics,
    prepare_dataset,
)


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def append_results(results_file: Path, row: dict[str, Any]) -> None:
    results_file.parent.mkdir(parents=True, exist_ok=True)
    file_exists = results_file.exists()
    with results_file.open('a', newline='', encoding='utf-8') as handle:
        writer = csv.DictWriter(handle, fieldnames=RESULT_COLUMNS)
        if not file_exists:
            writer.writeheader()
        writer.writerow({column: row.get(column, '') for column in RESULT_COLUMNS})


def configure_official_peft(model: torch.nn.Module, method: str, rank: int, alpha: float, dropout: float):
    if method == 'ft':
        for param in model.parameters():
            param.requires_grad = True
        return model, dropout

    effective_dropout = 0.0 if method == 'dora' else dropout
    peft_config = LoraConfig(
        task_type=TaskType.SEQ_CLS,
        r=rank,
        lora_alpha=alpha,
        lora_dropout=effective_dropout,
        target_modules=TARGET_MODULES,
        bias='none',
        modules_to_save=['classifier'],
        use_dora=(method == 'dora'),
    )
    peft_model = get_peft_model(model, peft_config)
    peft_model.print_trainable_parameters()
    return peft_model, effective_dropout


def make_training_args(output_dir: Path, method: str, learning_rate: float, batch_size: int, eval_batch_size: int, epochs: float, seed: int):
    kwargs = {
        'output_dir': str(output_dir),
        'learning_rate': learning_rate,
        'per_device_train_batch_size': batch_size,
        'per_device_eval_batch_size': eval_batch_size,
        'num_train_epochs': epochs,
        'save_strategy': 'epoch' if method == 'ft' else 'no',
        'load_best_model_at_end': method == 'ft',
        'metric_for_best_model': 'macro_f1',
        'greater_is_better': True,
        'logging_steps': 50,
        'report_to': [],
        'seed': seed,
    }
    arg_names = inspect.signature(TrainingArguments.__init__).parameters
    if 'eval_strategy' in arg_names:
        kwargs['eval_strategy'] = 'epoch'
    else:
        kwargs['evaluation_strategy'] = 'epoch'
    return TrainingArguments(**kwargs)


def run_official_peft_training(
    method: str,
    seed: int,
    rank: int = 8,
    alpha: float | None = None,
    dropout: float = 0.05,
    dataset_name: str = DATASET,
    task_type: str = 'single_label',
    label_map: str | None = None,
    label_threshold: float = 0.5,
    epochs: float = 5.0,
    learning_rate: float | None = None,
    batch_size: int = 16,
    eval_batch_size: int = 32,
    max_length: int = 256,
    max_train_samples: int | None = None,
    max_eval_samples: int | None = None,
) -> dict[str, Any]:
    set_seed(seed)
    if alpha is None and method in {'lora', 'dora'}:
        alpha = 2 * rank
    if learning_rate is None:
        learning_rate = 2e-5 if method == 'ft' else 2e-4

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)
    dataset, num_labels, label_map_dict = prepare_dataset(
        str(ROOT / dataset_name),
        tokenizer,
        max_length,
        max_train_samples,
        max_eval_samples,
        task_type=task_type,
        label_map_path=label_map,
    )
    id2label = {idx: label for label, idx in label_map_dict.items()}
    problem_type = 'multi_label_classification' if task_type == 'multi_label' else None
    model = build_phobert_classifier(
        MODEL_NAME,
        num_labels=num_labels,
        problem_type=problem_type,
        id2label=id2label,
        label2id=label_map_dict,
    )
    model, effective_dropout = configure_official_peft(model, method, rank, alpha or 0, dropout)
    counts = count_parameters(model)

    rank_part = f'_r{rank}' if method in {'lora', 'dora'} else ''
    run_id = f'official_{method}{rank_part}_seed{seed}_{int(time.time())}'
    output_dir = OUTPUT_ROOT / run_id
    training_args = make_training_args(output_dir, method, learning_rate, batch_size, eval_batch_size, epochs, seed)

    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    start = time.perf_counter()
    trainer_kwargs = {
        'model': model,
        'args': training_args,
        'train_dataset': dataset['train'],
        'eval_dataset': dataset['validation'],
        'data_collator': DataCollatorWithPadding(tokenizer),
        'compute_metrics': make_multilabel_compute_metrics(label_threshold) if task_type == 'multi_label' else compute_metrics,
    }
    trainer_arg_names = inspect.signature(Trainer.__init__).parameters
    if 'processing_class' in trainer_arg_names:
        trainer_kwargs['processing_class'] = tokenizer
    else:
        trainer_kwargs['tokenizer'] = tokenizer
    trainer = Trainer(**trainer_kwargs)
    trainer.train()
    metrics = trainer.evaluate()
    train_time = time.perf_counter() - start
    peak_vram_mb = torch.cuda.max_memory_allocated() / (1024 * 1024) if torch.cuda.is_available() else 0.0

    output_dir.mkdir(parents=True, exist_ok=True)
    if method == 'ft':
        trainer.save_model(str(output_dir / 'checkpoint_final'))
    else:
        model.save_pretrained(str(output_dir / 'adapter_checkpoint'))

    row = {
        'run_id': run_id,
        'method': method,
        'model_name': MODEL_NAME,
        'dataset': dataset_name,
        'task_type': task_type,
        'rank': '' if method == 'ft' else rank,
        'alpha': '' if method == 'ft' else alpha,
        'dropout': '' if method == 'ft' else effective_dropout,
        'seed': seed,
        'init_magnitude': '',
        'use_detached_gradient': '',
        'accuracy': metrics.get('eval_accuracy'),
        'micro_f1': metrics.get('eval_micro_f1'),
        'macro_f1': metrics.get('eval_macro_f1'),
        'weighted_f1': metrics.get('eval_weighted_f1'),
        'samples_f1': metrics.get('eval_samples_f1'),
        'hamming_loss': metrics.get('eval_hamming_loss'),
        'label_threshold': label_threshold if task_type == 'multi_label' else '',
        'trainable_params': counts.trainable,
        'total_params': counts.total,
        'trainable_percent': counts.trainable_percent,
        'train_time_sec': train_time,
        'peak_vram_mb': peak_vram_mb,
        'checkpoint_size_mb': directory_size_mb(output_dir),
        'output_dir': str(output_dir.relative_to(ROOT) if output_dir.is_relative_to(ROOT) else output_dir),
    }
    with (output_dir / 'metrics.json').open('w', encoding='utf-8') as handle:
        json.dump(row, handle, indent=2, ensure_ascii=False)
    with (output_dir / 'config.json').open('w', encoding='utf-8') as handle:
        json.dump(
            {
                'method': method,
                'model_name': MODEL_NAME,
                'dataset': dataset_name,
                'task_type': task_type,
                'rank': rank,
                'alpha': alpha,
                'dropout': effective_dropout,
                'seed': seed,
                'target_modules': TARGET_MODULES,
                'official_peft': method in {'lora', 'dora'},
                'use_dora': method == 'dora',
                'modules_to_save': ['classifier'] if method in {'lora', 'dora'} else [],
                'num_labels': num_labels,
                'label_map': label_map_dict,
            },
            handle,
            indent=2,
            ensure_ascii=False,
        )
    append_results(RESULTS, row)
    print(json.dumps(row, indent=2, ensure_ascii=False))
    return row

## Run Benchmark

In [ ]:
RUN_DIRECT_CHECK = True
RUN_TRAINING = False
TRAIN_MODE = 'benchmark'  # 'single', 'smoke', or 'benchmark'
seeds = [97, 98, 99]
ranks = [8, 16]

def build_run_specs():
    specs = []
    for seed in seeds:
        specs.append({'method': 'ft', 'seed': seed})
        for method, rank in itertools.product(['lora', 'dora'], ranks):
            specs.append({'method': method, 'seed': seed, 'rank': rank, 'alpha': 2 * rank, 'dropout': 0.05})
    return specs

benchmark_specs = build_run_specs()
if TRAIN_MODE == 'single':
    run_specs = [{'method': 'dora', 'seed': 42, 'rank': 8, 'alpha': 16, 'dropout': 0.05, 'epochs': 1, 'max_train_samples': 64, 'max_eval_samples': 64}]
elif TRAIN_MODE == 'smoke':
    run_specs = [{**spec, 'epochs': 1, 'max_train_samples': 64, 'max_eval_samples': 64} for spec in benchmark_specs[:3]]
elif TRAIN_MODE == 'benchmark':
    run_specs = benchmark_specs
else:
    raise ValueError("TRAIN_MODE must be 'single', 'smoke', or 'benchmark'")

if RUN_DIRECT_CHECK:
    assert RESULTS.name == 'offical_benchmark_result.csv'
    assert RESULT_COLUMNS == list(__import__('pandas').read_csv(REFERENCE_RESULTS, nrows=0).columns)
    assert DATASET_PATH.exists(), f'Missing dataset: {DATASET_PATH}'
    print('Direct check passed.')
    print('Rows planned:', len(run_specs))
    for spec in run_specs:
        print(spec)

if RUN_TRAINING:
    for spec in run_specs:
        run_official_peft_training(**spec)
else:
    print('Set RUN_TRAINING = True to execute the training specs above.')

## Aggregate Results

In [ ]:
import pandas as pd
from IPython.display import display

df = pd.read_csv(RESULTS) if RESULTS.exists() else pd.DataFrame(columns=RESULT_COLUMNS)
if df.empty or df.dropna(how='all').empty:
    print(f'No benchmark rows found yet: {RESULTS}')
    print('Set RUN_TRAINING = True above, then rerun this notebook.')
else:
    assert list(df.columns) == RESULT_COLUMNS
    df = df.dropna(subset=['method'], how='any')
display(df)

In [ ]:
if df.empty:
    summary = pd.DataFrame()
else:
    summary = (
        df.groupby(['method', 'rank'], dropna=False)
        .agg(
            accuracy_mean=('accuracy', 'mean'), accuracy_std=('accuracy', 'std'),
            macro_f1_mean=('macro_f1', 'mean'), macro_f1_std=('macro_f1', 'std'),
            weighted_f1_mean=('weighted_f1', 'mean'), weighted_f1_std=('weighted_f1', 'std'),
            trainable_percent_mean=('trainable_percent', 'mean'),
            peak_vram_mb_mean=('peak_vram_mb', 'mean'),
            train_time_sec_mean=('train_time_sec', 'mean'),
            checkpoint_size_mb_mean=('checkpoint_size_mb', 'mean'),
            runs=('run_id', 'count'),
        )
        .reset_index()
        .sort_values(['method', 'rank'])
    )
display(summary)

## Plots

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if df.empty:
    print('No benchmark rows to plot yet.')
else:
    plot_df = df.copy()
    plot_df['rank_label'] = plot_df['rank'].fillna('full').astype(str)
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    sns.barplot(data=plot_df, x='method', y='macro_f1', hue='rank_label', ax=axes[0])
    axes[0].set_title('Macro-F1')
    sns.barplot(data=plot_df, x='method', y='trainable_params', hue='rank_label', ax=axes[1])
    axes[1].set_title('Trainable params')
    sns.barplot(data=plot_df, x='method', y='peak_vram_mb', hue='rank_label', ax=axes[2])
    axes[2].set_title('Peak VRAM (MB)')
    plt.tight_layout()